### Bloque 3 · Expositor 3
## Modelos supervisados: riesgo de mora e inspecciones

**Facturación residencial de energía eléctrica, 2023–2024**

**Objetivo:** pasar de los segmentos a dos decisiones operativas: qué cuentas terminarán en mora grave (4.5, árbol de decisión) y a cuáles conviene inspeccionar primero (4.6, Random Forest).

- **Entrada:** `data/datos_limpios.parquet` generado por el Bloque 1.
- **Salida:** `data/top_1000_cuentas_inspeccion.csv` con las cuentas no inspeccionadas de mayor riesgo.
- **Unidad de análisis:** la cuenta. El árbol predice mora de más de 60 días en el segundo semestre de 2024; el bosque predice el resultado de una inspección técnica.
- **Regla:** nunca se usan datos posteriores al momento de la predicción; el corte temporal está en el código, no en el azar.

**Mapa de gráficas (en orden):**

| # | Gráfica | Pregunta que responde |
|---|---|---|
| G1 | Árbol de decisión | ¿Qué reglas aprende el modelo para mora > 60 días? |
| G2 | Matriz de confusión y métricas (1×2) | ¿Cómo se equivoca el árbol en datos no vistos? |
| G3 | Riesgo por incremento de factura (1×1) | ¿El alza anticipa la mora y pesa la región? |
| G4 | Importancia de variables (1×1) | ¿Qué señales usa el bosque para priorizar visitas? |
| G5 | Ranking de inspección (1×2) | ¿Cómo se distribuye el riesgo y cuánto mejora el top 20%? |

## Carga del parquet limpio

Solo se lee el archivo generado por el Bloque 1. Se valida que estén las columnas necesarias para no repetir la limpieza.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score, roc_curve, auc,
                             precision_score, recall_score, f1_score)
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 110,
})
PALETA = ['#2b5c8f', '#e4572e', '#4c9f70', '#f2b134', '#7b5ea7', '#66b3ff']
FIGSIZE = {'simple': (14, 5), 'doble': (15, 5), 'panel2x2': (15, 10), 'triple': (18, 5)}
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

CANDIDATOS_PARQUET = [Path('data/datos_limpios.parquet'),
                      Path('datos_limpios.parquet'),
                      Path('Energia/data/datos_limpios.parquet'),
                      Path('Energia/datos_limpios.parquet'),
                      Path('../data/datos_limpios.parquet'),
                      Path('../Energia/data/datos_limpios.parquet')]
RUTA_DATOS = next((ruta for ruta in CANDIDATOS_PARQUET if ruta.is_file()), None)
if RUTA_DATOS is None:
    raise FileNotFoundError(
        'No se encontró datos_limpios.parquet. Ejecuta primero 01_limpieza_y_exploratorio.ipynb '
        'o verifica que exista Energia/data/datos_limpios.parquet en el repositorio.'
    )

print(f'Parquet: {RUTA_DATOS.resolve()}')
df = pd.read_parquet(RUTA_DATOS)
requeridas = ['id_cuenta', 'periodo', 'estrato', 'operador', 'tipo_medidor', 'consumo_kwh',
              'valor_total_cop', 'dias_mora', 'lectura_estimada', 'autogeneracion_solar',
              'resultado_inspeccion']
faltantes = [c for c in requeridas if c not in df.columns]
if faltantes:
    raise ValueError(f'El parquet no tiene las columnas esperadas: {faltantes}. Vuelve a ejecutar la 01.')

print(f'Filas: {len(df):,} | Cuentas: {df["id_cuenta"].nunique():,}')
print(f'Periodos: {df["periodo"].min()} a {df["periodo"].max()}')
print('Este notebook solo lee el parquet: no vuelve a limpiar el CSV ni lo sobrescribe.')

## Fase 3.7 / Problemática 4.5 · ¿Qué cuentas terminarán con mora de más de 60 días?

Se arma una tabla con una fila por cuenta usando **solo información hasta 2024-06**: consumo y facturación promedio de 2023, meses con mora en 2023 y factura promedio del primer semestre de 2024. El objetivo es si la cuenta acumuló **mora > 60 días en 2024-07..12**.

Se revisa el balance de clases antes de entrenar y se trata con `class_weight='balanced'`. Se ajusta un árbol poco profundo (`max_depth=4`, `min_samples_leaf=500`) para que las reglas quepan en una página y se puedan leer como criterios de negocio.

In [ ]:
from sklearn.metrics import confusion_matrix

hist = df[df['periodo'] <= '2024-06']
futuro = df[df['periodo'] >= '2024-07']
assert hist['periodo'].max() <= '2024-06' and futuro['periodo'].min() >= '2024-07'
print(f'Corte temporal: se entrena con {hist["periodo"].min()} a {hist["periodo"].max()} y el '
      f'objetivo se mide en {futuro["periodo"].min()} a {futuro["periodo"].max()}.')
print('Revisión de fuga: ninguna variable de entrada usa meses de julio a diciembre de 2024.')

y_mora = futuro.groupby('id_cuenta')['dias_mora'].apply(lambda x: int((x > 60).any()))

f_2023 = hist[hist['periodo'].str.startswith('2023')].groupby('id_cuenta').agg(
    cons_2023=('consumo_kwh', 'mean'),
    val_2023=('valor_total_cop', 'mean'),
    meses_mora_2023=('dias_mora', lambda x: (x > 0).sum())
)
f_24s1 = hist[hist['periodo'] >= '2024-01'].groupby('id_cuenta').agg(
    val_24s1=('valor_total_cop', 'mean')
)
perfil_base = hist.groupby('id_cuenta').agg(
    estrato=('estrato', 'first'),
    operador=('operador', 'first'),
    tipo_medidor=('tipo_medidor', 'first')
)

df_arbol = perfil_base.join([f_2023, f_24s1, y_mora.rename('target_mora_60')]).dropna()
df_arbol['inc_factura_pct'] = ((df_arbol['val_24s1'] - df_arbol['val_2023']) / df_arbol['val_2023']) * 100

# Las cuentas con facturación 2023 en cero generan infinitos, que scikit-learn rechaza.
n_inf = int(np.isinf(df_arbol['inc_factura_pct']).sum())
df_arbol['inc_factura_pct'] = df_arbol['inc_factura_pct'].replace([np.inf, -np.inf], np.nan)
df_arbol = df_arbol.dropna(subset=['inc_factura_pct']).copy()
print(f'Cuentas con facturación 2023 en cero (se descartan): {n_inf}')
print(f'Cuentas modelables: {len(df_arbol):,} | con mora > 60 en 2024-07..12: '
      f'{df_arbol["target_mora_60"].mean() * 100:.2f}%')

# Balance de clases ANTES de entrenar y tratamiento decidido
positivos = int(df_arbol['target_mora_60'].sum())
negativos = len(df_arbol) - positivos
print(f'Balance de clases: {positivos:,} positivos vs {negativos:,} negativos '
      f'(1 positivo por cada {negativos / positivos:.1f} negativos).')
print('  Tratamiento: class_weight="balanced", que sube el peso de la clase minoritaria durante el '
      'entrenamiento. No se remuestrea el conjunto, para que las proporciones que se reportan sigan '
      'siendo las reales.')

X_arb = pd.get_dummies(df_arbol[['estrato', 'operador', 'tipo_medidor', 'cons_2023', 'val_2023', 'meses_mora_2023', 'inc_factura_pct']], drop_first=True)
y_arb = df_arbol['target_mora_60']

X_train, X_test, y_train, y_test = train_test_split(X_arb, y_arb, test_size=0.3, stratify=y_arb, random_state=42)

clf_tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=500, class_weight='balanced', random_state=42)
cv_res = cross_validate(clf_tree, X_train, y_train, cv=5, scoring=['precision', 'recall', 'f1', 'roc_auc'])
print('\nValidación cruzada de 5 particiones (sobre el 70% de entrenamiento):')
for m in ['test_precision', 'test_recall', 'test_f1', 'test_roc_auc']:
    print(f'  {m:>15}: {cv_res[m].mean():.3f}  (desv. {cv_res[m].std():.3f})')
print(f'  Énfasis en sensibilidad: el árbol detecta {cv_res["test_recall"].mean() * 100:.1f}% de las cuentas '
      'que sí terminan en mora grave. La precisión es baja porque solo 1 de cada '
      f'{negativos / positivos:.0f} cuentas cae en la clase positiva; para una alerta temprana se prefiere '
      'no dejar morosos por fuera aunque se visiten algunos falsos positivos.')

clf_tree.fit(X_train, y_train)

# Nombres legibles para la imagen y para las reglas
ETIQUETAS = {
    'estrato': 'Estrato',
    'cons_2023': 'Consumo medio 2023',
    'val_2023': 'Factura media 2023',
    'meses_mora_2023': 'Meses con mora 2023',
    'inc_factura_pct': 'Incremento factura 2024-S1 vs 2023 (%)',
}
nombres_arbol = [ETIQUETAS.get(c, c.replace('operador_', 'Operador: ').replace('tipo_medidor_', 'Medidor: '))
                 for c in X_arb.columns]

### Corte temporal y balance de clases

El modelo se entrena solo con meses hasta 2024-06 y se evalúa con lo que ocurre de 2024-07 en adelante. La lectura rápida siguiente cuantifica el desbalance de la etiqueta que se quiere predecir.

In [ ]:
print('Lectura rápida:')
print(f'  La clase positiva es {positivos:,} de {len(df_arbol):,} cuentas '
      f'({positivos / len(df_arbol) * 100:.2f}%).')
print('  Ese desbalance es la razón de usar class_weight="balanced" y de mirar sensibilidad, no exactitud.')

**Lectura del corte.** El corte es estricto: el modelo solo ve meses hasta 2024-06 y se evalúa con lo que pasa de 2024-07 en adelante. La etiqueta está desbalanceada —la mora grave es una minoría—, así que la exactitud sería engañosa: un modelo que dijera "nadie cae en mora" acertaría casi siempre y no serviría. Por eso se reporta sensibilidad y se pondera la clase minoritaria.

### G1 · Árbol de decisión

**Pregunta:** ¿qué reglas simples y legibles separan a las cuentas que terminan en mora grave?

In [ ]:
fig, ax = plt.subplots(figsize=(22, 11))
plot_tree(clf_tree, feature_names=nombres_arbol, class_names=['Sin mora > 60', 'Mora > 60'],
          filled=True, rounded=True, impurity=False, proportion=False, fontsize=9, ax=ax)
fig.suptitle('G1 · Árbol de decisión: mora > 60 días en 2024-07..12 (profundidad 4)',
             fontsize=16, fontweight='bold', y=0.97)
fig.text(0.5, 0.012,
         'Nota: "samples" es el número real de cuentas del nodo. Como el modelo usa '
         'class_weight="balanced", los "value" y los colores están ponderados por clase.',
         ha='center', fontsize=10, style='italic', color='#444444')
plt.tight_layout(rect=[0, 0.035, 1, 0.945])
plt.show()

print('Lectura rápida:')
print('  Cada camino de la raíz a una hoja es una regla "si ... entonces ...".')
print('  El árbol es poco profundo a propósito: se prioriza que la regla se pueda explicar en una reunión.')

**Lectura G1.** El árbol se lee de arriba hacia abajo: cada corte es una condición y cada hoja es un grupo de cuentas con una tasa de mora. La profundidad 4 y la hoja mínima de 500 buscan reglas estables, no un ajuste perfecto. Como el modelo usa `class_weight="balanced"`, el tamaño del nodo (`samples`) es real pero los colores están ponderados: sirven para leer la regla, no para estimar la proporción real de morosos.

### G2 · Matriz de confusión y métricas

**Pregunta:** ¿cómo se equivoca el árbol en el 30% de cuentas que no vio?

In [ ]:
y_pred_tree = clf_tree.predict(X_test)
proba_tree = clf_tree.predict_proba(X_test)[:, 1]

matriz = confusion_matrix(y_test, y_pred_tree, labels=[0, 1])
display(pd.DataFrame(matriz,
                     index=['Real: sin mora > 60', 'Real: mora > 60'],
                     columns=['Predicho: sin mora > 60', 'Predicho: mora > 60']))

print('\nMétricas en el 30% de prueba (cuentas no vistas):')
print(f'  Precisión    : {precision_score(y_test, y_pred_tree):.3f}')
print(f'  Sensibilidad : {recall_score(y_test, y_pred_tree):.3f}')
print(f'  F1           : {f1_score(y_test, y_pred_tree):.3f}')
print(f'  ROC AUC      : {roc_auc_score(y_test, proba_tree):.3f}')
print(f'  Exactitud    : {(y_pred_tree == y_test).mean():.3f} (no sirve sola: solo '
      f'{y_test.mean() * 100:.2f}% del prueba es positivo)')

print('\nReporte por clase:')
print(classification_report(y_test, y_pred_tree, target_names=['Sin mora > 60', 'Mora > 60']))

metricas = pd.Series({
    'Precisión': precision_score(y_test, y_pred_tree),
    'Sensibilidad': recall_score(y_test, y_pred_tree),
    'F1': f1_score(y_test, y_pred_tree),
    'ROC AUC': roc_auc_score(y_test, proba_tree),
})

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE['doble'])
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Predicho\nsin mora', 'Predicho\nmora > 60'],
            yticklabels=['Real\nsin mora', 'Real\nmora > 60'])
axes[0].set_title('Matriz de confusión (30% de prueba)')

metricas.plot.bar(ax=axes[1], color=PALETA[0], rot=0)
axes[1].set_ylim(0, 1)
axes[1].set_title('Métricas del árbol en prueba')
axes[1].set_ylabel('Valor')
for i, valor in enumerate(metricas.values):
    axes[1].text(i, valor + 0.02, f'{valor:.2f}', ha='center', fontsize=10)

fig.suptitle('G2 · Desempeño del árbol en cuentas no vistas', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Lectura G2.** La matriz de confusión muestra el intercambio entre falsos positivos y falsos negativos. En una alerta temprana de mora, el error caro es el falso negativo: dejar por fuera a una cuenta que sí va a caer en mora grave. Por eso la sensibilidad es la métrica que manda; la precisión baja es el costo de no perder morosos, y se acepta porque visitar unos falsos positivos es más barato que un cobro jurídico.

### Reglas de negocio por hoja (tabla de apoyo)

**Pregunta:** ¿qué hojas concentran la mora grave y en qué orden se deben atender?

In [ ]:
def reglas_lenguaje_natural(modelo, nombres, X, y, max_reglas=5, min_cuentas=500):
    """Devuelve las hojas con mayor tasa de mora como reglas 'si ... entonces ...'."""
    arbol = modelo.tree_

    def recorrer(nodo, condiciones, salida):
        if arbol.children_left[nodo] == -1:
            salida[nodo] = condiciones
            return
        izq, der = arbol.children_left[nodo], arbol.children_right[nodo]
        nombre = nombres[arbol.feature[nodo]]
        umbral = arbol.threshold[nodo]
        recorrer(izq, condiciones + [(nombre, '<=', umbral)], salida)
        recorrer(der, condiciones + [(nombre, '>', umbral)], salida)

    rutas = {}
    recorrer(0, [], rutas)

    def texto_condicion(nombre, operador, umbral):
        if umbral == 0.5 and nombre.startswith(('Operador: ', 'Medidor: ')):
            categoria = nombre.split(': ', 1)[1]
            return f'es {categoria}' if operador == '>' else f'no es {categoria}'
        if nombre in ('Estrato', 'Meses con mora 2023'):
            if operador == '<=':
                return f'{nombre.lower()} ≤ {int(np.floor(umbral))}'
            return f'{nombre.lower()} ≥ {int(np.floor(umbral)) + 1}'
        return f'{nombre} {operador} {umbral:,.1f}'

    def compactar(condiciones):
        """Une varios cortes sobre la misma variable en un solo rango legible."""
        limites, orden = {}, []
        for nombre, operador, umbral in condiciones:
            if nombre not in limites:
                limites[nombre] = {'inf': None, 'sup': None}
                orden.append(nombre)
            if operador == '<=':
                actual = limites[nombre]['sup']
                limites[nombre]['sup'] = umbral if actual is None else min(actual, umbral)
            else:
                actual = limites[nombre]['inf']
                limites[nombre]['inf'] = umbral if actual is None else max(actual, umbral)
        partes = []
        for nombre in orden:
            inf, sup = limites[nombre]['inf'], limites[nombre]['sup']
            if inf is not None:
                partes.append(texto_condicion(nombre, '>', inf))
            if sup is not None:
                partes.append(texto_condicion(nombre, '<=', sup))
        return partes

    asignadas = modelo.apply(X)
    tabla = (pd.DataFrame({'hoja': asignadas, 'y': np.asarray(y)})
             .groupby('hoja')['y'].agg(Cuentas='size', Positivas='sum'))
    tabla['Tasa_%'] = tabla['Positivas'] / tabla['Cuentas'] * 100
    tabla = tabla[tabla['Cuentas'] >= min_cuentas].sort_values('Tasa_%', ascending=False)

    filas = []
    for hoja, fila in tabla.head(max_reglas).iterrows():
        condiciones = ' y '.join(compactar(rutas[hoja]))
        filas.append({
            'Regla': f'Si {condiciones}, entonces',
            'Cuentas': int(fila['Cuentas']),
            'En mora grave': int(fila['Positivas']),
            'Tasa mora grave %': round(fila['Tasa_%'], 2),
        })
    return pd.DataFrame(filas)


reglas_45 = reglas_lenguaje_natural(clf_tree, nombres_arbol, X_arb, y_arb)
display(reglas_45)
print('\nReglas de negocio (proporción real de mora grave en cada hoja):')
for i, fila in reglas_45.iterrows():
    print(f'  {i + 1}. {fila["Regla"]} el {fila["Tasa mora grave %"]:.2f}% de esas cuentas '
          f'({fila["En mora grave"]:,} de {fila["Cuentas"]:,}) termina en mora grave.')

print('\nÁrbol completo en texto (cada línea es un corte; sirve para auditar las reglas):')
print(export_text(clf_tree, feature_names=nombres_arbol))

**Lectura de las reglas.** Las reglas se ordenan por tasa real de mora grave, no por tamaño: una hoja pequeña con tasa muy alta puede ser más accionable que una hoja grande con tasa moderada. La tabla es el entregable de la problemática 4.5: cada fila se lee como "si se cumplen estas condiciones, entonces este porcentaje de esas cuentas termina en mora grave". El texto del árbol permite auditar de dónde sale cada condición.

### G3 · Riesgo por incremento de factura: Caribe vs resto

**Pregunta:** ¿el alza de la factura anticipa la mora y el efecto es distinto en el Caribe?

In [ ]:
df_arbol['region'] = np.where(df_arbol['operador'].isin(['AIR-E', 'AFINIA']), 'Caribe', 'Resto del país')
df_arbol['tramo_inc'] = pd.cut(df_arbol['inc_factura_pct'], bins=[-np.inf, 0, 15, 30, np.inf],
                               labels=['<=0%', '0-15%', '15-30%', '>30%'])
tabla_region = (pd.crosstab(df_arbol['tramo_inc'], df_arbol['region'],
                            values=df_arbol['target_mora_60'], aggfunc='mean') * 100).round(2)
display(tabla_region)
conteo_region = pd.crosstab(df_arbol['tramo_inc'], df_arbol['region'])
display(conteo_region)

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(tabla_region, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
ax.set_title('G3 · % de cuentas con mora > 60 días por tramo de alza y región')
ax.set_xlabel('Región')
ax.set_ylabel('Incremento de la factura 2024-S1 vs 2023')
plt.tight_layout()
plt.show()

print('Lectura de la tabla de riesgo (4.5):')
print('  Cada celda es el % de cuentas que terminó con mora > 60 días, por tramo de incremento de la')
print('  factura y por región. Si el % sube con el tramo, el alza anticipa la mora; si el Caribe está')
print('  por encima del resto en todos los tramos, el efecto es de la región y no solo del alza.')
difs = (tabla_region['Caribe'] - tabla_region['Resto del país']).dropna()
if len(difs) and (difs > 0).all():
    print(f'  El Caribe está por encima del resto en los {len(difs)} tramos, con una diferencia de '
          f'{difs.min():.2f} a {difs.max():.2f} pp: la región pesa por su cuenta y no solo la '
          'magnitud del alza.')
elif len(difs):
    print(f'  La diferencia Caribe - resto va de {difs.min():.2f} a {difs.max():.2f} pp; no es '
          'uniforme en todos los tramos, así que el efecto de la región depende del tamaño del alza.')

**Lectura G3.** Si la tasa de mora sube con el tramo de incremento, el alza de la factura funciona como señal temprana. Si además el Caribe está por encima del resto en todos los tramos, la región pesa por su cuenta y no solo la magnitud del alza. Esa lectura separa dos acciones: política tarifaria (alza) y acompañamiento regional (Caribe), y evita atribuir a una sola causa lo que en realidad son dos.

## Fase 3.8 / Problemática 4.6 · ¿A qué cuentas conviene inspeccionar?

Cada cuenta inspeccionada aporta una etiqueta (`Anomalía confirmada` / `Sin novedad`). Para no filtrar el futuro, los predictores se calculan en la **ventana de 12 meses anterior a la inspección**: nivel y variabilidad del consumo, caída entre los primeros y los últimos seis meses, meses con consumo casi nulo, proporción de lecturas estimadas y autogeneración. Las cuentas no inspeccionadas se puntúan con sus **últimos 12 meses**.

Se prueba un conjunto pequeño y documentado de hiperparámetros, se entrena un **Random Forest de 250 árboles** y se compara contra un árbol base usando **la misma partición**. Luego se ordenan las cuentas **no inspeccionadas** por probabilidad de anomalía, se guarda el top 1.000 y se reporta la precisión del 20% de mayor probabilidad frente a la tasa histórica de acierto.

In [ ]:
from sklearn.model_selection import cross_val_score

inspecciones = df[df['resultado_inspeccion'].notna()].sort_values('periodo').groupby('id_cuenta').last().reset_index()
tasa_historica = (inspecciones['resultado_inspeccion'] == 'Anomalía confirmada').mean() * 100
print(f"Tasa de acierto histórica actual: {tasa_historica:.2f}%")
print(f'Cuentas inspeccionadas: {len(inspecciones):,} (una inspección por cuenta)')

df_sorted = df.sort_values(['id_cuenta', 'periodo'])
df_sorted['periodo_idx'] = df_sorted['periodo'].str[:4].astype(int) * 12 + df_sorted['periodo'].str[5:7].astype(int)

insp_map = inspecciones.set_index('id_cuenta')['periodo'].apply(lambda p: int(p[:4]) * 12 + int(p[5:7])).to_dict()
# Las cuentas sin inspección usan todo el histórico disponible (corte en 2025).
df_sorted['corte_idx'] = df_sorted['id_cuenta'].map(insp_map).fillna(2025 * 12 + 1)

ventana_12m = df_sorted[(df_sorted['periodo_idx'] >= df_sorted['corte_idx'] - 12) &
                        (df_sorted['periodo_idx'] < df_sorted['corte_idx'])].copy()
# Posición del mes dentro de la ventana (1 = el más antiguo de los 12).
ventana_12m['mes_rel'] = ventana_12m['periodo_idx'] - (ventana_12m['corte_idx'] - 12)

# Revisión de fuga: el mes de la inspección y los posteriores quedan fuera por construcción.
assert (ventana_12m['periodo_idx'] < ventana_12m['corte_idx']).all()
assert ventana_12m.groupby('id_cuenta')['periodo_idx'].max().le(
    ventana_12m.groupby('id_cuenta')['corte_idx'].max() - 1).all()
print('Revisión de fuga: los predictores usan los 12 meses estrictamente anteriores a la '
      'inspección; no entra el mes de la visita ni los meses posteriores, cuando una anomalía '
      'confirmada ya habría normalizado el consumo.')

p_primeros = ventana_12m[ventana_12m['mes_rel'] <= 6].groupby('id_cuenta')['consumo_kwh'].mean()
p_ultimos = ventana_12m[ventana_12m['mes_rel'] > 6].groupby('id_cuenta')['consumo_kwh'].mean()

feats_rf = ventana_12m.groupby('id_cuenta').agg(
    cons_std=('consumo_kwh', 'std'),
    cons_mean=('consumo_kwh', 'mean'),
    meses_nulos=('consumo_kwh', lambda x: (x < 10).sum()),
    prop_estimadas=('lectura_estimada', lambda x: (x == 'S').mean()),
    autogeneracion=('autogeneracion_solar', lambda x: int((x == 'S').any())),
    estrato=('estrato', 'first'),
    operador=('operador', 'first'),
    tipo_medidor=('tipo_medidor', 'first')
)
feats_rf['caida_consumo_pct'] = ((p_ultimos - p_primeros) / (p_primeros + 1e-5)) * 100
feats_rf['cv_consumo'] = (feats_rf['cons_std'] / (feats_rf['cons_mean'] + 1e-5))

feats_encoded = pd.get_dummies(feats_rf.drop(columns=['cons_std', 'cons_mean']), drop_first=True).fillna(0)

ids_insp = inspecciones['id_cuenta']
X_insp = feats_encoded.loc[feats_encoded.index.isin(ids_insp)]
y_insp = (inspecciones.set_index('id_cuenta').loc[X_insp.index, 'resultado_inspeccion']
          .map({'Anomalía confirmada': 1, 'Sin novedad': 0}).astype(int))
X_no_insp = feats_encoded.loc[~feats_encoded.index.isin(ids_insp)]

print(f'\nCuentas inspeccionadas con ventana de 12 meses: {len(X_insp):,} '
      f'(se excluyen {len(ids_insp) - len(X_insp):,} sin historia previa suficiente)')
print(f'Positivos en la muestra de inspección: {int(y_insp.sum()):,} de {len(y_insp):,} '
      f'({y_insp.mean() * 100:.2f}%)')
print(f'Cuentas no inspeccionadas por priorizar: {len(X_no_insp):,}')

Xi_train, Xi_test, yi_train, yi_test = train_test_split(X_insp, y_insp, test_size=0.30, stratify=y_insp, random_state=42)
print(f'Partición: {len(Xi_train):,} cuentas para entrenar y {len(Xi_test):,} para probar (30%, '
      'estratificada por clase). El árbol base y el bosque usan EXACTAMENTE esta misma partición, '
      'así la comparación de la tabla es justa.')

# ---------- Ajuste documentado de hiperparámetros ----------
configs_rf = [
    ('profundidad 6, hoja 15', {'max_depth': 6, 'min_samples_leaf': 15}),
    ('profundidad 8, hoja 15', {'max_depth': 8, 'min_samples_leaf': 15}),
    ('profundidad 8, hoja 30', {'max_depth': 8, 'min_samples_leaf': 30}),
    ('profundidad 12, hoja 15', {'max_depth': 12, 'min_samples_leaf': 15}),
]
filas_ajuste = []
for nombre, params in configs_rf:
    modelo = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                    random_state=42, n_jobs=-1, **params)
    auc_cv = cross_val_score(modelo, Xi_train, yi_train, cv=3, scoring='roc_auc', n_jobs=-1).mean()
    filas_ajuste.append({'Configuración': nombre, 'ROC_AUC_CV_3fold': round(auc_cv, 4), **params})

ajuste_rf = pd.DataFrame(filas_ajuste).sort_values('ROC_AUC_CV_3fold', ascending=False)
display(ajuste_rf)
mejor_params = {k: ajuste_rf.iloc[0][k] for k in ('max_depth', 'min_samples_leaf')}
mejor_params = {k: (None if pd.isna(v) else int(v)) for k, v in mejor_params.items()}
print(f'Se probaron 4 combinaciones de profundidad máxima y tamaño mínimo de hoja (validación '
      f'cruzada de 3 particiones sobre el entrenamiento). Se conserva la de mayor ROC AUC: '
      f'max_depth={mejor_params["max_depth"]}, min_samples_leaf={mejor_params["min_samples_leaf"]}, '
      'con 250 árboles.')

dt_base = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, class_weight='balanced', random_state=42).fit(Xi_train, yi_train)
rf_model = RandomForestClassifier(n_estimators=250, class_weight='balanced', random_state=42,
                                  n_jobs=-1, **mejor_params).fit(Xi_train, yi_train)

comparativa = []
for nombre, mod in [('Árbol Base', dt_base), ('Random Forest', rf_model)]:
    preds = mod.predict(Xi_test)
    probs = mod.predict_proba(Xi_test)[:, 1]
    comparativa.append({
        'Modelo': nombre,
        'Precision': round(precision_score(yi_test, preds), 3),
        'Sensibilidad': round(recall_score(yi_test, preds), 3),
        'F1': round(f1_score(yi_test, preds), 3),
        'ROC_AUC': round(roc_auc_score(yi_test, probs), 3)
    })
comparativa = pd.DataFrame(comparativa)
display(comparativa)

probs_rf = rf_model.predict_proba(Xi_test)[:, 1]
umbral_top20 = np.percentile(probs_rf, 80)
precision_top20 = yi_test[probs_rf >= umbral_top20].mean() * 100
print(f"Precisión Top 20% mayor probabilidad: {precision_top20:.2f}%")
print(f'  La tasa histórica es {tasa_historica:.2f}%: concentrar las visitas en el 20% de mayor '
      f'probabilidad la llevaría a {precision_top20:.2f}%, es decir, '
      f'{precision_top20 - tasa_historica:+.2f} puntos porcentuales '
      f'({precision_top20 / tasa_historica:.2f} veces la tasa actual).')
print('  La mejora es grande; por eso se revisó primero la fuga (arriba): el modelo solo ve meses')
print('  anteriores a la inspección. Parte de la explicación es que las inspecciones se concentraron')
print('  en cuentas con irregularidades persistentes, visibles en su historial. La cifra debe')
print('  confirmarse con una muestra aleatoria de visitas antes de prometerla como resultado.')

### Comparación de modelos: AUC y métricas

El área bajo la curva ROC (AUC) resume la capacidad de ranking de cada modelo en un solo número, que es lo que necesita una lista de inspección. La tabla siguiente compara las métricas del árbol base y del Random Forest sobre la misma partición de prueba.

In [ ]:
probs_dt = dt_base.predict_proba(Xi_test)[:, 1]
probs_rf = rf_model.predict_proba(Xi_test)[:, 1]
fpr_dt, tpr_dt, _ = roc_curve(yi_test, probs_dt)
fpr_rf, tpr_rf, _ = roc_curve(yi_test, probs_rf)
auc_dt = auc(fpr_dt, tpr_dt)
auc_rf = auc(fpr_rf, tpr_rf)

print('Lectura rápida:')
print(f'  AUC árbol base: {auc_dt:.3f} | AUC bosque: {auc_rf:.3f} '
      f'(diferencia {auc_rf - auc_dt:+.3f}).')
print('  Cuanto más alto el AUC, mejor ordena el modelo las cuentas sospechosas.')
display(comparativa)

**Lectura de la comparación.** El AUC no depende de un umbral: resume el intercambio entre detectar anomalías y generar falsas alarmas en todos los cortes posibles. Si el bosque supera al árbol base, ordena mejor las cuentas y conviene para armar la lista de inspección; si las cifras se acercan, el árbol gana por transparencia. Un AUC cercano a 0.5 equivaldría al azar, así que ambos modelos deben quedar claramente por encima.

### G4 · Importancia de variables

**Pregunta:** ¿qué señales del historial usan los modelos para priorizar una inspección?

In [ ]:
perm = permutation_importance(rf_model, Xi_test, yi_test, n_repeats=10, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({
    'Variable': X_insp.columns,
    'Importancia_Impureza': rf_model.feature_importances_,
    'Importancia_Permutacion': perm.importances_mean,
    'Desv_Permutacion': perm.importances_std,
}).sort_values('Importancia_Permutacion', ascending=False)
display(imp_df.head(10))

top_imp = imp_df.head(10).sort_values('Importancia_Permutacion', ascending=False)
fig, ax = plt.subplots(figsize=FIGSIZE['simple'])
sns.barplot(data=top_imp, x='Variable', y='Importancia_Permutacion', ax=ax,
            color=PALETA[0], yerr=top_imp['Desv_Permutacion'])
ax.set_title('G4 · Importancia por permutación (caída del ROC AUC al desordenar la variable)')
ax.set_xlabel('')
ax.set_ylabel('Caída media del ROC AUC')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

LECTURA_VAR = {
    'caida_consumo_pct': 'una caída fuerte del consumo sugiere abandono del servicio o una conexión irregular',
    'cv_consumo': 'mucha variabilidad mes a mes es típica de consumos irregulares o de lecturas estimadas',
    'meses_nulos': 'los meses con consumo casi nulo apuntan a cuentas inactivas o a manipulación del medidor',
    'prop_estimadas': 'la proporción de lecturas estimadas dice cuánta historia real hay detrás de la cuenta',
    'autogeneracion': 'la autogeneración cambia el patrón de consumo esperado y puede ocultar el consumo real',
    'Estrato': 'el estrato resume el nivel socioeconómico y el tipo de vivienda',
}
print('\nSeñales más importantes según permutación (cuánto cae el ROC AUC al desordenarlas):')
for _, fila in imp_df.head(3).iterrows():
    lectura = LECTURA_VAR.get(fila['Variable'],
                              'aporta información propia del perfil de la cuenta')
    print(f'  - {fila["Variable"]}: -{fila["Importancia_Permutacion"]:.3f} de ROC AUC; {lectura}.')

print('\nPor qué la importancia por impureza y la de permutación pueden no coincidir:')
print('  La de impureza se acumula durante el entrenamiento y premia variables que permiten muchos')
print('  cortes; favorece variables continuas y de alta cardinalidad, y no mide si el modelo generaliza.')
print('  La de permutación se calcula después, sobre el conjunto de prueba, desordenando una variable y')
print('  midiendo cuánto pierde el desempeño. Castiga variables redundantes y refleja el aporte real.')
print('  Importancia no es causalidad: una variable importante es útil para predecir, no la causa del')
print('  resultado. Aquí el bosque aprende de inspecciones, no de un experimento.')

**Lectura G4.** Las barras responden "cuánto pierde el modelo si esta variable se desordena". Las variables de irregularidad del consumo —caída, variabilidad y meses nulos— suelen encabezar la lista porque describen el patrón que los inspectores ya buscaban. La importancia por impureza se lee aparte: describe cómo se construyó el bosque, no cuánto aporta a la generalización. Y ninguna de las dos es causalidad: son señales predictivas, no causas de la anomalía.

### G5 · Ranking de inspección

**Pregunta:** ¿cómo se distribuye el riesgo entre las cuentas no inspeccionadas y cuánto mejora el top 20%?

In [ ]:
probs_no_insp = rf_model.predict_proba(X_no_insp)[:, 1]
top_1000 = pd.DataFrame({'id_cuenta': X_no_insp.index, 'probabilidad_anomalia': probs_no_insp}).sort_values('probabilidad_anomalia', ascending=False).head(1000)
RUTA_SALIDA = RUTA_DATOS.with_name('top_1000_cuentas_inspeccion.csv')
top_1000.to_csv(RUTA_SALIDA, index=False)
print(f'\nTop 1.000 guardado en {RUTA_SALIDA} ({len(top_1000):,} filas)')
print(f'  Rango de probabilidad del top: {top_1000["probabilidad_anomalia"].min():.3f} a '
      f'{top_1000["probabilidad_anomalia"].max():.3f}. Media del ranking completo: '
      f'{probs_no_insp.mean():.3f}.')

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE['doble'])
sns.histplot(probs_no_insp, bins=50, ax=axes[0], color=PALETA[0])
corte_top = top_1000['probabilidad_anomalia'].min()
axes[0].axvline(corte_top, color=PALETA[1], linestyle='--', linewidth=2,
                label=f'Corte top 1.000 ({corte_top:.2f})')
axes[0].set_title('Distribución de probabilidad de anomalía (cuentas no inspeccionadas)')
axes[0].set_xlabel('Probabilidad de anomalía')
axes[0].set_ylabel('Cuentas')
axes[0].legend(fontsize=8)

axes[1].bar(['Tasa histórica', 'Top 20%'], [tasa_historica, precision_top20],
            color=[PALETA[0], PALETA[1]])
for i, valor in enumerate([tasa_historica, precision_top20]):
    axes[1].text(i, valor, f'{valor:.2f}%', ha='center', va='bottom', fontsize=11)
axes[1].set_title('Acierto histórico vs. precisión del Top 20%')
axes[1].set_ylabel('% de anomalías confirmadas')
axes[1].set_ylim(0, max(tasa_historica, precision_top20) * 1.25)

fig.suptitle('G5 · Ranking de inspección y mejora sobre la tasa histórica', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nAdvertencia de sesgo (obligatoria en 4.6):')
print('  Las inspecciones históricas NO se hicieron al azar: se visitó a las cuentas que ya parecían')
print('  sospechosas con criterios informales. Por eso:')
print(f'  - La tasa de acierto de {tasa_historica:.2f}% describe a esa población sesgada, no al universo '
      'de cuentas.')
print('  - El modelo hereda el sesgo: puede estar aprendiendo "lo que los inspectores ya buscaban" y no')
print('    todas las anomalías. Las cuentas nunca visitadas no tienen etiqueta y podrían ser positivas.')
print('  - La precisión del Top 20% es una estimación dentro de la muestra inspeccionada; para')
print('    generalizarla conviene validar el ranking con una muestra aleatoria de visitas.')

print('\nLectura rápida:')
print(f'  Con 250 árboles y 30% de prueba, el bosque alcanza ROC-AUC = {comparativa.iloc[1]["ROC_AUC"]:.3f} '
      f'frente a {comparativa.iloc[0]["ROC_AUC"]:.3f} del árbol base.')
print('  La precisión del Top 20% se compara contra la tasa histórica de acierto impresa arriba:')
print('  si es mayor, concentrar las visitas en ese 20% encuentra más anomalías que visitar al azar.')

**Lectura G5.** El histograma muestra que la mayoría de las cuentas no inspeccionadas tiene probabilidad baja, pero una cola pequeña concentra el riesgo. El corte del top 1.000 es la lista operativa: se ordena de mayor a menor probabilidad y se carga en la programación de visitas. La comparación contra la tasa histórica responde la pregunta de valor: si el top 20% acierta más que visitar al azar, el modelo sirve para asignar recursos, no solo para describir.

## Cierre del Bloque 3

- **Entregable 4.5:** reglas de negocio legibles y validadas en un corte temporal estricto.
- **Entregable 4.6:** `data/top_1000_cuentas_inspeccion.csv` ordenado por probabilidad de anomalía.
- **Advertencia transversal:** las etiquetas de inspección no son aleatorias; el ranking debe validarse con una muestra aleatoria de visitas antes de prometer la mejora como resultado.